# Shadow Fuzzer Run Analysis

Comprehensive analysis of a single Shadow simulation run.

In [ ]:
# Papermill parameters — injected by render_notebooks.py
run_dir = ""
run_id = ""

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on path for imports
repo_root = Path(run_dir).resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from notebooks.utils import load_run_data

data = load_run_data(run_dir)
metadata = data["metadata"]
stats = data["stats"]
regions = data["regions"]
bandwidths = data["bandwidths"]
run_id = run_id or data["run_id"]

print(f"Analyzing run: {run_id}")
print(f"Run directory: {run_dir}")

## 1. Run Overview

In [ ]:
# Build overview table
sim = metadata.get("simulation", {})
fuzzer_cfg = metadata.get("fuzzer", {})
node_counts = metadata.get("node_counts", {})

overview = {
    "Run ID": run_id,
    "Total Nodes": sim.get("total_nodes", "N/A"),
    "Duration (s)": fuzzer_cfg.get("duration_secs", "N/A"),
    "Runner": fuzzer_cfg.get("runner", "N/A"),
    "Seed": fuzzer_cfg.get("seed", "N/A"),
    "Total Subnets": sim.get("total_subnets", "N/A"),
}

for client, count in sorted(node_counts.items()):
    overview[f"Nodes ({client})"] = count

pd.DataFrame([overview]).T.rename(columns={0: "Value"})

In [ ]:
# Client distribution pie chart
if node_counts:
    fig = px.pie(
        names=list(node_counts.keys()),
        values=list(node_counts.values()),
        title=f"Client Distribution ({sum(node_counts.values())} nodes)",
        hole=0.4,
    )
    fig.update_traces(textposition="inside", textinfo="percent+label+value")
    fig.show()
else:
    print("No node count data available.")

In [ ]:
# Region and bandwidth distribution
from collections import Counter

region_counts = Counter(regions.values()) if regions else Counter()
bw_counts = Counter(bandwidths.values()) if bandwidths else Counter()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Region Distribution", "Bandwidth Distribution"),
)

if region_counts:
    fig.add_bar(
        x=list(region_counts.keys()),
        y=list(region_counts.values()),
        name="Regions",
        marker_color="steelblue",
        row=1, col=1,
    )

if bw_counts:
    fig.add_bar(
        x=list(bw_counts.keys()),
        y=list(bw_counts.values()),
        name="Bandwidth",
        marker_color="coral",
        row=1, col=2,
    )

fig.update_layout(showlegend=False, height=400)
fig.show()

## 2. Block Propagation

In [ ]:
blocks = stats.get("blocks", {})
block_slots = blocks.get("slots", [])
block_summary = blocks.get("summary", {})

if block_summary.get("warning"):
    print(f"\u26a0\ufe0f {block_summary['warning']}")

if not block_slots:
    print("No block data available.")
else:
    print(f"Blocks published: {block_summary.get('n_published', 0)}")
    print(f"Blocks received: {block_summary.get('n_received', 0)}")
    print(f"Slots with block size data: {block_summary.get('n_with_block_size', 0)}")
    print(f"Slots with block data: {len(block_slots)}")

In [ ]:
# Block propagation latency scatter plot
if block_slots:
    rows = []
    for s in block_slots:
        pub_ms = s.get("published_ms")
        if pub_ms is None:
            continue
        for host, recv_ms in s.get("receive_timestamps_ms", {}).items():
            rows.append({
                "slot": s["slot"],
                "host": host,
                "client": host.rsplit("-", 1)[0] if "-" in host else host,
                "latency_ms": round(recv_ms - pub_ms, 1),
            })

    if rows:
        df = pd.DataFrame(rows)
        n_hosts = df["host"].nunique()

        fig = px.scatter(
            df, x="slot", y="latency_ms",
            color="client" if n_hosts > 20 else "host",
            title="Block Propagation Latency (publish \u2192 receive)",
            labels={"latency_ms": "Latency (ms)", "slot": "Slot", "client": "Client"},
        )
        fig.update_layout(
            height=500,
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=-0.35 if n_hosts <= 20 else -0.2,
                xanchor="center",
                x=0.5,
                font=dict(size=9),
            ),
            margin=dict(b=80),
        )
        fig.show()
    else:
        print("No publish\u2192receive latency data available.")

In [ ]:
# Block propagation latency percentiles per slot
if "df" in globals() and not df.empty:
    df_pct = (
        df.groupby("slot")["latency_ms"]
        .quantile([0.50, 0.95, 0.99])
        .unstack()
        .rename(columns={0.50: "p50", 0.95: "p95", 0.99: "p99"})
        .reset_index()
    )

    fig = go.Figure()
    for col in ["p50", "p95", "p99"]:
        fig.add_scatter(
            x=df_pct["slot"],
            y=df_pct[col],
            mode="lines+markers",
            name=col,
        )

    fig.update_layout(
        title="Block Propagation Latency Percentiles per Slot",
        xaxis_title="Slot",
        yaxis_title="Latency (ms)",
        height=450,
    )
    fig.show()
else:
    print("No publish\u2192receive latency data available.")

In [ ]:
# Block size per slot
block_size_rows = [
    {"slot": s["slot"], "block_size_kib": s["block_size_bytes"] / 1024}
    for s in block_slots
    if "block_size_bytes" in s
]

if block_size_rows:
    df_block_sizes = pd.DataFrame(block_size_rows)
    fig = px.line(
        df_block_sizes,
        x="slot",
        y="block_size_kib",
        markers=True,
        title="Block Size per Slot",
        labels={"slot": "Slot", "block_size_kib": "Block size (KiB)"},
    )
    fig.update_layout(height=400)
    fig.show()
else:
    print("No block size data available.")

## 3. Attestation Coverage

In [ ]:
attestations = stats.get("attestations", {})
coverage = attestations.get("coverage", {})
cov_slots = coverage.get("slots", [])
cov_summary = coverage.get("summary", {})

if cov_summary.get("warning"):
    print(f"\u26a0\ufe0f {cov_summary['warning']}")

if not cov_slots:
    print("No attestation coverage data available.")
else:
    print(f"Slots with coverage data: {cov_summary.get('slots_with_data', 0)}")
    for field in ["median_slot_p50_nodes_to_95_attestations_ms",
                  "median_slot_p90_nodes_to_95_attestations_ms",
                  "median_slot_p95_nodes_to_95_attestations_ms"]:
        if field in cov_summary:
            label = field.replace("median_slot_", "").replace("_nodes_to_95_attestations_ms", "")
            print(f"  Median {label}: {cov_summary[field]} ms")

In [ ]:
# Attestation coverage per slot
if cov_slots:
    df_cov = pd.DataFrame(cov_slots)
    n_nodes = df_cov["n_nodes"].iloc[0] if len(df_cov) > 0 else 0

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    df_cov["coverage_ratio"] = df_cov["n_nodes_reached_threshold"] / n_nodes
    fig.add_scatter(
        x=df_cov["slot"], y=df_cov["coverage_ratio"],
        mode="lines+markers", name="Coverage",
        line=dict(color="steelblue"),
        secondary_y=False,
    )

    if "p95_nodes_to_95_attestations_ms" in df_cov.columns:
        fig.add_scatter(
            x=df_cov["slot"], y=df_cov["p95_nodes_to_95_attestations_ms"],
            mode="lines+markers", name="p95 latency",
            line=dict(color="coral", dash="dot"),
            secondary_y=True,
        )

    fig.add_hline(y=0.95, line_dash="dash", line_color="green",
                  annotation_text="95% target", secondary_y=False)

    fig.update_yaxes(title_text="Node Coverage Ratio", secondary_y=False)
    fig.update_yaxes(title_text="Latency (ms)", secondary_y=True)
    fig.update_xaxes(title_text="Slot")
    fig.update_layout(title="Attestation Coverage per Slot", height=450)
    fig.show()

## 4. Chain Finality

In [ ]:
chain = stats.get("chain_status", {})
chain_slots = chain.get("slots", [])
chain_summary = chain.get("summary", {})

if chain_summary.get("warning"):
    print(f"\u26a0\ufe0f {chain_summary['warning']}")

if not chain_slots:
    print("No chain status data available.")
else:
    print(f"Slots with chain data: {chain_summary.get('slots_with_data', 0)}")
    print(f"Hosts with chain data: {chain_summary.get('hosts_with_data', 0)}")

In [ ]:
# Chain finality heatmaps
if chain_slots:
    all_hosts = sorted({h for s in chain_slots for h in s.get("hosts", {})})
    n_hosts = len(all_hosts)

    for metric, key, colorscale, title in [
        ("head", "head_slot", "Blues", "Chain Head Slot per Node"),
        ("finalized", "latest_finalized_slot", "Greens", "Finalized Slot per Node"),
    ]:
        rows = []
        for s in chain_slots:
            slot = s["slot"]
            for host in all_hosts:
                hs = s.get("hosts", {}).get(host, {})
                rows.append({"slot": slot, "host": host, metric: hs.get(key, 0)})

        df_m = pd.DataFrame(rows)

        if n_hosts > 30:
            # Aggregate by client type for large runs
            df_m["client"] = df_m["host"].str.rsplit("-", n=1).str[0]
            agg = df_m.groupby(["slot", "client"])[metric].mean().reset_index()
            pivot = agg.pivot(index="client", columns="slot", values=metric)
            y_title = "Client (avg)"
            tick_size = 10
        else:
            pivot = df_m.pivot(index="host", columns="slot", values=metric)
            y_title = "Node"
            tick_size = 9 if n_hosts <= 16 else 8

        fig = go.Figure(data=go.Heatmap(
            z=pivot.values,
            x=[str(c) for c in pivot.columns],
            y=list(pivot.index),
            colorscale=colorscale,
        ))
        fig.update_layout(
            title=f"{title}{' (avg per client)' if n_hosts > 30 else ''}",
            xaxis_title="Slot", yaxis_title=y_title,
            height=max(400, len(pivot.index) * 25),
            margin=dict(l=max(80, max(len(str(y)) for y in pivot.index) * 7)),
            yaxis=dict(tickfont=dict(size=tick_size)),
        )
        fig.show()

## 5. Network Topology

In [ ]:
topo_path = data.get("topology_gml_path")

if topo_path and topo_path.exists():
    try:
        import networkx as nx
        G = nx.read_gml(topo_path)
        print(f"Topology: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

        pos = nx.spring_layout(G, seed=42)

        edge_x = []
        edge_y = []
        for u, v in G.edges():
            x0, y0 = pos[u]
            x1, y1 = pos[v]
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])

        node_x = [pos[n][0] for n in G.nodes()]
        node_y = [pos[n][1] for n in G.nodes()]

        node_labels = [str(n) for n in G.nodes()]
        node_colors = []
        for n in G.nodes():
            label = str(n)
            if label in regions:
                node_colors.append(regions[label])
            else:
                node_colors.append("unknown")

        fig = go.Figure()
        fig.add_scatter(x=edge_x, y=edge_y, mode="lines",
                        line=dict(width=0.5, color="#888"), hoverinfo="none")
        fig.add_scatter(
            x=node_x, y=node_y, mode="markers",
            marker=dict(size=8, color=node_colors, colorscale="Set1", showscale=True),
            text=[f"{l} ({c})" for l, c in zip(node_labels, node_colors)],
            hoverinfo="text",
        )
        fig.update_layout(
            title="Network Topology",
            showlegend=False, height=600,
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        )
        fig.show()
    except ImportError:
        print("networkx not installed. Install with: pip install networkx")
else:
    print("No topology.gml file found.")

In [ ]:
# Warnings
warnings = stats.get("warnings", [])
if warnings:
    print(f"\u26a0\ufe0f {len(warnings)} warnings:")
    for w in warnings:
        print(f"  - {w}")
else:
    print("\u2705 No warnings.")